In [3]:
import os
import pdfplumber
import re
import json
import pandas as pd

TENDERS_ROOT = "tenders"  # Root directory containing tender documents

# Patterns for extracting key sections
EVALUATION_SECTION_PATTERN = r"(?i)(Evaluation and Award|Evaluation|Award Criteria|Evaluation & Award|EVALUATION AND AWARD)(.*?)((?:\n[A-Z][a-z]*|\Z))"

# Only capture annexes with "Technical Specifications" or "Tender Specifications"
ANNEX_SECTION_PATTERN = r"(?i)(Annex.*?(Technical Specifications|Tender Specifications))(.*?)((?:\n[A-Z][a-z]*|\Z))"

SUBSECTIONS = ["Exclusion Criteria", "Selection Criteria", "Award Criteria"]

def extract_text_and_tables_from_pdf(pdf_path):
    """Extracts text and tables from a PDF file."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = "\n".join([page.extract_text() for page in pdf.pages if page.extract_text()])
            
            # Extract tables
            all_tables = []
            for page in pdf.pages:
                for table in page.extract_tables():
                    if table:
                        all_tables.append(table)
        
        return text, all_tables
    except Exception as e:
        print(f"⚠️ Error processing {pdf_path}: {str(e)}")
        return None, None

def extract_section(text, pattern):
    """Extracts a section based on a given regex pattern, safely handling missing groups."""
    if not text:
        return "Not found"
    
    matches = re.findall(pattern, text, re.DOTALL)
    
    # Ensure we return the correct group index based on available data
    extracted_sections = [match[-2].strip() if len(match) > 2 else match[1].strip() for match in matches]

    return "\n".join(extracted_sections) if extracted_sections else "Not found"

def process_pdfs():
    """Recursively finds and processes all PDFs in the tenders directory."""
    extracted_data = {}
    skipped_files = []

    for root, _, files in os.walk(TENDERS_ROOT):
        if "tender_specifications.pdf" in files:
            tender_folder = os.path.basename(root)
            pdf_path = os.path.join(root, "tender_specifications.pdf")

            text, tables = extract_text_and_tables_from_pdf(pdf_path)

            if text:
                structured_data = {
                    "Evaluation and Award": extract_section(text, EVALUATION_SECTION_PATTERN),
                    "Annex - Technical/Tender Specifications": extract_section(text, ANNEX_SECTION_PATTERN),
                    "Tables": tables if tables else "No tables found"
                }

                # Extract subsections within Evaluation and Award
                for subsection in SUBSECTIONS:
                    subsection_pattern = rf"(?i)({subsection})(.*?)(?=\n[A-Z][a-z]*|\Z)"
                    structured_data[subsection] = extract_section(text, subsection_pattern)

                extracted_data[tender_folder] = structured_data
            else:
                skipped_files.append(pdf_path)

    # Save structured data to JSON
    with open("structured_evaluation_and_technical_annexes.json", "w", encoding="utf-8") as f:
        json.dump(extracted_data, f, indent=4, ensure_ascii=False)

    # Save skipped files
    with open("skipped_files.log", "w", encoding="utf-8") as f:
        f.write("\n".join(skipped_files))

    print("✅ Extraction completed. Data saved in structured_evaluation_and_technical_annexes.json.")
    print(f"⚠️ {len(skipped_files)} PDFs were skipped. Check skipped_files.log.")

if __name__ == "__main__":
    process_pdfs()


⚠️ Error processing tenders/Cyber Threat Intelligence support services/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/EU4 Security Moldova Project_ Cyber crime lab for Republic of Moldova/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/CFT-1765 Supply of office furniture/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/Report on the Role of the Labour Inspectorate and Prevention Services in Supporting OSH Compliance in Portugal (Lot 1), in Poland (Lot 2), and in Ireland (Lot 3)/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/Supply of Customised Promotional Items/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/Sweden – Support Services for the Organisation of Events and Other Communication and Information Activities of the Liaison Office/tender_speci

Data-loss while decompressing corrupted data


⚠️ Error processing tenders/Construction of the Distribution Networks in Buchanan and Greenville Towns, Liberia/tender_specifications.pdf: Unexpected EOF
⚠️ Error processing tenders/Translation and revision services for Eurojust external projects/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/Front and Back-office Reception Services/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/Support to Water Statistics Production and Ecosystem Accounting/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/Framework Contract on Communication Outreach Activities: Lot 1 – EURES Communication Activities and Lot 2 – ELA Communication and Outreach Activities/tender_specifications.pdf: No /Root object! - Is this really a PDF?
⚠️ Error processing tenders/Provision of Floor Staff for the Europa Experience Multimedia Centre in Prague/tender_specifications.pdf: No /Root